# Aquaplanet with customized initial conditions

This notebook demonstrates how to change configuration of an aquaplanet simulation.

The code will be largely the same until the section "Customize Initial Condition" and "Run Couple Model".

In [ ]:
from pathlib import Path

import jcm
from jcm.physics.speedy.speedy_coords import get_speedy_coords
import jax_datetime as jdt

from jem.base.coupler import Coupler
from jem.components import JCMComponent, SlabOceanModel, SlabSeaiceModel
from jem.components.slab import SlabGrid

use_ipython = 'get_ipython' in globals()

## Configurations

In [ ]:
start_datetime = jdt.to_datetime("2000-01-01")
coupling_timestep = jdt.to_timedelta(1, "day")
simulation_name = "01-02_aquaplanet_customized_initial_condition"
output_dir = (Path("output") / simulation_name).resolve()
output_dir.mkdir(exist_ok=True, parents=True)
output_figures = {
    "initial_condition": output_dir / "initial_sst.png",
    "animation": output_dir / "animation_humidity_sst.gif",
}

## Creating Flux and Scalar Exchange between Components

An *exchanger* is the only place where components exchange information. It is
traced with the rest of the coupled step, so it builds new carries with
`.replace(...)` instead of writing into the ones it is handed.

In [ ]:
def exchange(components, time):
    del time  # this exchange does not depend on the date

    atm = components["atm"]
    ocn = components["ocn"]
    seaice = components["seaice"]

    ocn = dict(ocn, forcing=ocn["forcing"].replace(
        total_heat_flux=atm["derived"].total_heat_flux,
    ))
    seaice = dict(seaice, forcing=seaice["forcing"].replace(
        ice_frazil_melt_energy=ocn["derived"].ice_frazil_melt_energy,
    ))
    atm = dict(atm, forcing=atm["forcing"].replace(
        sea_surface_temperature=ocn["state"].sea_surface_temperature,
        sice_am=seaice["derived"].ice_fraction,
    ))

    return dict(components, atm=atm, ocn=ocn, seaice=seaice)

## Create Components

In [ ]:
atm_model = jcm.model.Model(
    coords=get_speedy_coords(),  # T31 spectral resolution with 8 vertical levels
    start_date=start_datetime,
)

# Aquaplanet: no fractional mask, so every cell of the slab grid is ocean.
aquaplanet_grid = SlabGrid.from_coords(atm_model.coords.horizontal)

model = Coupler(
    dict(
        atm=JCMComponent(atm_model),
        ocn=SlabOceanModel(aquaplanet_grid),
        seaice=SlabSeaiceModel(aquaplanet_grid, name="seaice"),
    ),
    dict(exchange=exchange),
    coupling_timestep=coupling_timestep,
    start_date=start_datetime,
)

print(repr(model))

## Customize Initial Condition

`Coupler.initialize()` returns a `CoupledCarry`: one carry per component under
`.components`, plus the coupled step counter. It is a `flax.struct` dataclass and
the carries inside it are pytrees, so a customized initial condition is built by
*replacing* pieces of it rather than assigning into it.

In [ ]:
import jax.numpy as jnp

initial_coupled_carry = model.initialize()

ocean_model = model.components["ocn"]
sst_perturbation = (
    5
    * jnp.sin(ocean_model.grid.longitude_radian * 2)
    * jnp.cos(ocean_model.grid.latitude_radian) ** 3
)

ocean_carry = initial_coupled_carry.components["ocn"]
ocean_carry = dict(ocean_carry, state=ocean_carry["state"].replace(
    sea_surface_temperature=(
        ocean_carry["state"].sea_surface_temperature + sst_perturbation
    ),
))
customized_initial_coupled_carry = initial_coupled_carry.replace(
    components=dict(initial_coupled_carry.components, ocn=ocean_carry),
)

In [ ]:
import matplotlib as mplt
if not use_ipython:
    mplt.use("Agg")
import matplotlib.pyplot as plt


customized_sst = (
    customized_initial_coupled_carry.components["ocn"]["state"].sea_surface_temperature
)

lat = ocean_model.grid.latitude_radian[0, :] * 180.0 / jnp.pi
lon = ocean_model.grid.longitude_radian[:, 0] * 180.0 / jnp.pi
fig, ax = plt.subplots(1, 1)
mappable = ax.contourf(lon, lat, customized_sst.transpose() - 273.15, cmap="gnuplot")
cbar = plt.colorbar(mappable, ax=ax)
cbar.set_label("Customized initialial sea surface temperature [${}^\\circ \\mathrm{C}$]")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")

print(f"Saving initial condition figure: {output_figures['initial_condition']}")
plt.savefig(output_figures['initial_condition'], dpi=200)

if use_ipython:
    plt.show()

## Run Coupled Model

The trajectory function takes the initial carry as its argument, so the
customized one is simply what it is called with.

In [ ]:
simulation_interval = jdt.to_timedelta(60, "day")
run = model.generate_trajectory_function(
    int(simulation_interval / coupling_timestep)
)
final_carry, diagnostics = run(customized_initial_coupled_carry)

## Output into NetCDF

In [ ]:
output_dict = model.to_xarray(diagnostics)
output_dict_subsample = {}
subsample_skip = 5
for component_name, ds in output_dict.items():
    output_file = output_dir / f"{component_name:s}.nc"
    print(f"Output file: {str(output_file)}, with subsample_skip = {subsample_skip:d}")
    ds = ds.isel(time=slice(None, None, subsample_skip))
    ds.to_netcdf(output_file, engine="netcdf4")
    output_dict_subsample[component_name] = ds

## Visualization

In [ ]:

import matplotlib as mplt
if not use_ipython:
    mplt.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
import numpy as np

output_dict_animation = {
    component_name: _ds.isel(time=slice(None, None, 1))
    for component_name, _ds in output_dict_subsample.items()
}

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.gridlines(draw_labels=True)
cb = None
cf = None
cs = None
ch = None

def update(frame):
    print(f"Plotting frame={frame:d}")
    global cf, cb, cs, ch
    _data_q = output_dict_animation["atm"]["specific_humidity"].isel(time=frame, level=0)
    _data_sst = output_dict_animation["ocn"]["sea_surface_temperature"].isel(time=frame) - 273.15
    _data_sit = output_dict_animation["seaice"]["ice_thickness"].isel(time=frame)
    coords = _data_q.coords
    time_str = _data_q['time'].dt.strftime('%Y-%m-%d').to_numpy().item()
    lat = coords["lat"]
    lon = coords["lon"]

    # Remove previous frame's artists before drawing the new ones
    cf and cf.remove()
    cs and cs.remove()
    ch and ch.remove()
    
    # Plot the humidity field for the current time step
    cyclic_data_q, cyclic_lon = add_cyclic_point(_data_q.to_numpy().transpose(), coord=lon)
    mappable = ax.contourf(
        cyclic_lon, lat,
        cyclic_data_q,
        levels=1 + np.linspace(0, 1, 21) * 10,
        transform=ccrs.PlateCarree(), 
        cmap='GnBu',
        extend="both",
    )
    
    cyclic_data_sst, cyclic_lon = add_cyclic_point(_data_sst.to_numpy().transpose(), coord=lon)
    cs = ax.contour(
        cyclic_lon, lat,
        cyclic_data_sst,
        levels=np.arange(-2, 31, 4),
        transform=ccrs.PlateCarree(),
        colors="black",
    )
    ax.clabel(cs, fontsize=12)

    # Dot-hatch grid cells that carry any sea ice (thickness above zero)
    cyclic_data_sit, cyclic_lon = add_cyclic_point(_data_sit.to_numpy().transpose(), coord=lon)
    ch = ax.contourf(
        cyclic_lon, lat,
        cyclic_data_sit,
        levels=[1e-6, np.inf],
        colors="none",
        hatches=["."],
        transform=ccrs.PlateCarree(),
    )

    ax.set_title(f"[{time_str:s}]\nSurface specific humidity (shading) and sea surface temperature (contours, ${{}}^\\circ \\mathrm{{C}}$),\nwith sea ice (dotted hatching)")
    if cb is None:
        cb = plt.colorbar(ax=ax, mappable=mappable, orientation='vertical', shrink=0.7, pad=0.07)
        cb.set_label("[g/kg]", fontsize=12)
    
    return [cf,]
    
# Generate and save
ani = FuncAnimation(fig, update, frames=len(output_dict_animation["atm"].coords["time"]), interval=120, blit=False)
print("Saving animation: ", output_figures["animation"])
ani.save(output_figures["animation"], writer='pillow', dpi=200)

if use_ipython:
    from IPython.display import Image
    display(Image(output_figures["animation"]))